# Lay Betting Liability Compounding — Australian Racing

Simulates a compound lay-betting strategy on Betfair-style racing markets.

**Strategy**
- Start with a \$5 liability on the first race
- If the lay bet wins (runner does NOT win the race): compound 100% of balance into next liability
- If the lay bet loses (runner wins the race): bust — session ends
- Target: grow \$5 → \$1,000

**Runner selection** (Betfair minimum back-stake aware)
- Find the highest-odds runner where `back_stake = liability / (odds − 1) ≥ $2` (Betfair AU minimum)
- As balance grows, higher-odds runners become affordable → selection moves down the market
- Fallback to favourite when no runner satisfies the minimum back-stake constraint

**Market model**
- 101% overround (Betfair-style exchange)
- 5% commission on winning lay profit
- 5–24 runners per race
- Three race types: Thoroughbred, Harness, Greyhound (calibrated Gamma distributions)

**Simulation**
- 1,000 independent runs per race type
- Maximum 10,000 races per run

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## Parameters

In [ ]:
# ── Simulation parameters ────────────────────────────────────────────────────
N_SIMS            = 1_000      # independent runs per race type
MAX_RACES         = 10_000     # safety cap per run
STARTING_BALANCE  = 5.0        # first liability = starting balance
TARGET            = 1_000.0    # session ends when balance >= target

# ── Betfair parameters ───────────────────────────────────────────────────────
COMMISSION        = 0.05       # 5% on winning lay profit
OVERROUND         = 1.01       # 101% market overround
MIN_BACK_STAKE    = 2.0        # Betfair AU minimum back stake
MAX_LAY_ODDS      = 30.0       # hard cap on lay odds (exchange liquidity limit)

# ── Race parameters ──────────────────────────────────────────────────────────
MIN_RUNNERS       = 5
MAX_RUNNERS       = 24

# Race types: (label, Gamma shape, description)
# Gamma(α): lower α → more concentrated on favourite → higher fav win rate
RACE_TYPES = [
    ("Thoroughbred", 0.5, "~33% fav win rate"),
    ("Harness",      0.6, "~30% fav win rate"),
    ("Greyhound",    0.8, "~27% fav win rate"),
]

SEED = 42

## Core Functions

In [ ]:
def generate_race(n_runners: int, overround: float, shape: float, rng: np.random.Generator):
    """
    Generate a race market.

    Returns
    -------
    lay_odds  : np.ndarray, shape (n_runners,), sorted descending by true prob
                (index 0 = favourite)
    true_probs: np.ndarray, shape (n_runners,), sum to 1.0
    """
    raw = rng.gamma(shape, 1.0, size=n_runners)
    true_probs = raw / raw.sum()

    # Sort descending: index 0 is the favourite (highest true win prob)
    order = np.argsort(true_probs)[::-1]
    true_probs = true_probs[order]

    # Betfair-style: implied prob = true_prob × overround, lay_odds = 1 / implied_prob
    implied_probs = np.clip(true_probs * overround, 1e-6, 1.0)
    lay_odds = np.round(1.0 / implied_probs, 2)
    lay_odds = np.maximum(lay_odds, 1.01)

    return lay_odds, true_probs


def select_runner(
    lay_odds: np.ndarray,
    balance: float,
    min_back_stake: float = MIN_BACK_STAKE,
    max_lay_odds: float = MAX_LAY_ODDS,
):
    """
    Select the highest-odds runner affordable within Betfair constraints.

    A runner is affordable if the required back stake (liability / (odds - 1))
    is >= min_back_stake.  This rearranges to: odds <= 1 + balance / min_back_stake.

    Falls back to the favourite (index 0) if no runner satisfies the constraint.

    Returns
    -------
    (runner_index, selected_odds)
    """
    # Maximum affordable lay odds given min back stake requirement
    max_feasible = min(1.0 + balance / min_back_stake, max_lay_odds)

    best_idx  = None
    best_odds = 0.0

    for i, odds in enumerate(lay_odds):
        if 1.01 < odds <= max_feasible and odds > best_odds:
            best_odds = odds
            best_idx  = i

    # Fallback: favourite (always affordable as it has the shortest odds)
    if best_idx is None:
        best_idx  = 0
        best_odds = lay_odds[0]

    return best_idx, best_odds


def resolve_lay_bet(
    balance: float,
    lay_odds: float,
    true_prob_win: float,
    commission: float,
    rng: np.random.Generator,
):
    """
    Resolve a single lay bet.

    Lay bet mechanics:
      liability   = balance (100% all-in)
      back_stake  = liability / (lay_odds - 1)

      If runner DOESN'T win (lay wins):
        gross_profit = back_stake
        commission   = gross_profit * commission_rate
        new_balance  = balance + gross_profit - commission

      If runner WINS (lay loses):
        new_balance  = 0  → bust

    Returns
    -------
    (new_balance, lay_won, back_stake)
    """
    liability  = balance
    back_stake = liability / (lay_odds - 1.0)

    runner_wins = rng.random() < true_prob_win

    if runner_wins:
        # Lay loses: forfeit entire liability
        return 0.0, False, back_stake
    else:
        # Lay wins: collect back_stake minus commission
        gross_profit = back_stake
        net_profit   = gross_profit * (1.0 - commission)
        return balance + net_profit, True, back_stake

## Single-Run Simulator

In [ ]:
def run_simulation(
    shape: float,
    rng: np.random.Generator,
    starting_balance: float = STARTING_BALANCE,
    target: float = TARGET,
    overround: float = OVERROUND,
    commission: float = COMMISSION,
    min_runners: int = MIN_RUNNERS,
    max_runners: int = MAX_RUNNERS,
    min_back_stake: float = MIN_BACK_STAKE,
    max_lay_odds: float = MAX_LAY_ODDS,
    max_races: int = MAX_RACES,
):
    """
    Run a single simulation path.

    Returns a dict with outcome metrics.
    """
    balance         = starting_balance
    balance_path    = [balance]
    runner_ranks    = []    # which rank (0=fav, 1=2nd fav, ...) was selected each race
    race_count      = 0
    outcome         = "max_races"  # default if cap reached

    for _ in range(max_races):
        n_runners = rng.integers(min_runners, max_runners + 1)
        lay_odds, true_probs = generate_race(n_runners, overround, shape, rng)

        runner_idx, selected_odds = select_runner(
            lay_odds, balance, min_back_stake, max_lay_odds
        )
        runner_ranks.append(runner_idx)

        balance, lay_won, _ = resolve_lay_bet(
            balance, selected_odds, true_probs[runner_idx], commission, rng
        )
        race_count += 1
        balance_path.append(balance)

        if balance <= 0.0:
            outcome = "bust"
            break
        if balance >= target:
            outcome = "target"
            break

    return {
        "outcome":       outcome,
        "race_count":    race_count,
        "final_balance": balance,
        "balance_path":  np.array(balance_path),
        "runner_ranks":  runner_ranks,
    }

## Monte Carlo — All Race Types

In [ ]:
results_by_type = {}

for label, shape, desc in RACE_TYPES:
    rng = np.random.default_rng(SEED)
    sims = [run_simulation(shape, rng) for _ in range(N_SIMS)]
    results_by_type[label] = sims
    busts   = sum(1 for s in sims if s["outcome"] == "bust")
    targets = sum(1 for s in sims if s["outcome"] == "target")
    print(f"{label} ({desc}): {busts} busts, {targets} targets out of {N_SIMS}")

## Summary Statistics

In [ ]:
summary_rows = []

for label, shape, desc in RACE_TYPES:
    sims = results_by_type[label]
    n = len(sims)

    busts      = [s for s in sims if s["outcome"] == "bust"]
    targets    = [s for s in sims if s["outcome"] == "target"]
    max_races  = [s for s in sims if s["outcome"] == "max_races"]

    race_counts = [s["race_count"] for s in sims]
    bust_races  = [s["race_count"] for s in busts]   if busts   else [np.nan]
    tgt_races   = [s["race_count"] for s in targets] if targets else [np.nan]

    summary_rows.append({
        "Race Type":          label,
        "Gamma Shape":        shape,
        "Bust Count":         len(busts),
        "Target Count":       len(targets),
        "Max-Races Count":    len(max_races),
        "Bust Rate %":        f"{len(busts)/n*100:.1f}",
        "Target Rate %":      f"{len(targets)/n*100:.1f}",
        "Median Races (all)": f"{np.median(race_counts):.0f}",
        "Median Races (bust)":f"{np.median(bust_races):.0f}" if busts else "—",
        "Median Races (tgt)": f"{np.median(tgt_races):.0f}"  if targets else "—",
        "Mean Final Balance": f"${np.mean([s['final_balance'] for s in sims]):.2f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## Races-to-Bust / Races-to-Target Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle("Races-to-Bust Distribution (Bust sessions only)", fontsize=13)

colors = ["#e74c3c", "#3498db", "#2ecc71"]

for ax, (label, shape, desc), color in zip(axes, RACE_TYPES, colors):
    sims  = results_by_type[label]
    busts = [s["race_count"] for s in sims if s["outcome"] == "bust"]

    if busts:
        bins = range(0, min(max(busts) + 2, 50))
        ax.hist(busts, bins=bins, color=color, edgecolor="white", linewidth=0.5, alpha=0.85)
        ax.axvline(np.median(busts), color="black", linewidth=1.5,
                   linestyle="--", label=f"Median: {np.median(busts):.0f}")
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, "No busts", ha="center", va="center", transform=ax.transAxes)

    n_busts  = len(busts)
    n_total  = len(sims)
    ax.set_title(f"{label}\n({n_busts}/{n_total} bust, {n_busts/n_total*100:.1f}%)")
    ax.set_xlabel("Races before bust")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## Balance Paths — Sample Trajectories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Balance Paths — First 200 Simulations", fontsize=13)

colors = ["#e74c3c", "#3498db", "#2ecc71"]
N_SHOW = 200

for ax, (label, shape, desc), color in zip(axes, RACE_TYPES, colors):
    sims = results_by_type[label]

    for s in sims[:N_SHOW]:
        path = s["balance_path"]
        if s["outcome"] == "target":
            ax.plot(path, color="gold", alpha=0.9, linewidth=1.5, zorder=5)
        elif s["outcome"] == "bust":
            ax.plot(path, color=color, alpha=0.15, linewidth=0.7)
        else:
            ax.plot(path, color="grey", alpha=0.15, linewidth=0.7)

    ax.axhline(TARGET, color="black", linewidth=1, linestyle="--", label=f"Target ${TARGET:.0f}")
    ax.axhline(STARTING_BALANCE, color="grey", linewidth=0.8, linestyle=":")

    n_targets = sum(1 for s in sims[:N_SHOW] if s["outcome"] == "target")
    ax.set_title(f"{label}\n({n_targets} targets in first {N_SHOW} sims — gold)")
    ax.set_xlabel("Race number")
    ax.set_ylabel("Balance ($)")
    ax.set_yscale("log")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Runner Rank Selection — How Far Down the Market?

As balance grows, the selection algorithm automatically moves to higher-ranked (longer-odds) runners.
This shows the distribution of runner ranks selected across all races in all simulations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Runner Rank Selected (0 = Favourite)", fontsize=13)

colors = ["#e74c3c", "#3498db", "#2ecc71"]

for ax, (label, shape, desc), color in zip(axes, RACE_TYPES, colors):
    sims = results_by_type[label]

    all_ranks = []
    for s in sims:
        all_ranks.extend(s["runner_ranks"])

    if all_ranks:
        rank_counter = Counter(all_ranks)
        max_rank = max(rank_counter.keys())
        ranks  = list(range(max_rank + 1))
        counts = [rank_counter.get(r, 0) for r in ranks]
        total  = sum(counts)

        ax.bar(ranks, [c / total * 100 for c in counts],
               color=color, edgecolor="white", linewidth=0.5, alpha=0.85)

    ax.set_title(label)
    ax.set_xlabel("Runner rank (0 = favourite)")
    ax.set_ylabel("% of bets placed")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

## Balance Growth Factor per Win

Theoretical growth factor for each race type, showing how balance scales per successful lay bet at different odds levels.

In [ ]:
odds_range = np.linspace(1.05, 10.0, 300)

fig, ax = plt.subplots(figsize=(10, 4))

# Growth factor = new_balance / balance
# = (balance + back_stake * (1 - commission)) / balance
# = 1 + (1 / (odds - 1)) * (1 - commission)
growth = 1.0 + (1.0 / (odds_range - 1.0)) * (1.0 - COMMISSION)

ax.plot(odds_range, growth, color="steelblue", linewidth=2, label="Growth factor per win")

# Races needed to reach target: n = log(TARGET/START) / log(growth)
n_races_needed = np.log(TARGET / STARTING_BALANCE) / np.log(growth)

ax2 = ax.twinx()
ax2.plot(odds_range, n_races_needed, color="#e74c3c", linewidth=2, linestyle="--",
         label="Consecutive wins needed")
ax2.set_ylabel("Consecutive wins needed to reach target", color="#e74c3c")
ax2.tick_params(axis="y", labelcolor="#e74c3c")
ax2.set_ylim(0, 300)

ax.set_xlabel("Lay odds")
ax.set_ylabel("Balance growth factor per win")
ax.set_title(f"Lay Bet Compounding: Growth Factor vs Consecutive Wins Required\n"
             f"(Starting ${STARTING_BALANCE} → Target ${TARGET:.0f}, {COMMISSION*100:.0f}% commission)")
ax.set_xlim(1.05, 10.0)
ax.set_ylim(1.0, 3.0)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

ax.axvline(2.0, color="grey", linewidth=0.8, linestyle=":")
ax.text(2.05, 1.1, "Min back stake\nboundary", color="grey", fontsize=8)

plt.tight_layout()
plt.show()

# Print key values
for test_odds in [1.5, 2.0, 3.0, 5.0, 8.0, 10.0]:
    gf = 1.0 + (1.0/(test_odds - 1.0)) * (1.0 - COMMISSION)
    n  = np.log(TARGET/STARTING_BALANCE) / np.log(gf)
    # Probability of n consecutive wins (using avg win rate estimate)
    print(f"Odds {test_odds:.1f}: growth ×{gf:.4f}/win, needs {n:.0f} consecutive wins")

## Session Outcome Comparison — All Race Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Session Outcome Breakdown by Race Type", fontsize=13)

colors_map = {"bust": "#e74c3c", "target": "gold", "max_races": "#95a5a6"}
labels_map  = {"bust": "Bust (lost liability)",
               "target": f"Target reached (${TARGET:.0f})",
               "max_races": f"Max races ({MAX_RACES:,})"}

for ax, (label, shape, desc) in zip(axes, RACE_TYPES):
    sims = results_by_type[label]
    outcome_counts = Counter(s["outcome"] for s in sims)
    n = len(sims)

    outcomes = ["target", "max_races", "bust"]
    values   = [outcome_counts.get(o, 0) for o in outcomes]
    clrs     = [colors_map[o] for o in outcomes]
    lbls     = [f"{labels_map[o]}\n{outcome_counts.get(o,0)} ({outcome_counts.get(o,0)/n*100:.1f}%)"
                for o in outcomes]

    wedges, texts = ax.pie(
        values, labels=None, colors=clrs,
        startangle=90, counterclock=False,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )
    ax.legend(wedges, lbls, loc="lower center", fontsize=7.5,
              bbox_to_anchor=(0.5, -0.18))
    ax.set_title(f"{label}\n({desc})")

plt.tight_layout()
plt.show()

## Successful Target Runs — Deep Dive

In [ ]:
print("Target-Reaching Runs Detail")
print("=" * 70)

for label, shape, desc in RACE_TYPES:
    sims    = results_by_type[label]
    targets = [s for s in sims if s["outcome"] == "target"]

    print(f"\n{label} — {len(targets)} target run(s):")

    if not targets:
        print("  (none)")
        continue

    for i, s in enumerate(targets, 1):
        races   = s["race_count"]
        final   = s["final_balance"]
        ranks   = s["runner_ranks"]
        rank_c  = Counter(ranks)
        top3    = rank_c.most_common(3)
        print(f"  Run {i}: {races} races, final ${final:.2f} | "
              f"Top ranks: {top3}")

## Percentile Table — Final Balance

In [ ]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
rows = []

for label, shape, desc in RACE_TYPES:
    sims    = results_by_type[label]
    finals  = [s["final_balance"] for s in sims]
    row     = {"Race Type": label}
    for p in percentiles:
        row[f"p{p}"] = f"${np.percentile(finals, p):.2f}"
    rows.append(row)

pct_df = pd.DataFrame(rows)
print("Final Balance Percentiles")
print(pct_df.to_string(index=False))

## Key Findings

### Why the Success Rate Is So Low

The strategy requires compounding \$5 → \$1,000 (a **200× increase**) with 100% all-in staking.
Each bet is binary: win the lay → grow balance; lose the lay → lose everything.

At the typical early-stage selection (favourite, odds ≈ \$2.50–\$4.00):
- Growth per win ≈ ×1.24–×1.45
- Consecutive wins needed: **~25–50 races**
- Per-race survival probability (lay wins): ~75–85%
- Probability of 30 consecutive wins @ 80% per race: `0.80^30 ≈ 0.12%`

This matches the observed **~0.1–0.5%** success rate.

### Race Type Comparison
- **Thoroughbred**: highest favourite win rate (~33%) → more early-stage fallbacks to favourite → shorter odds → more wins needed → lower success
- **Harness**: intermediate
- **Greyhound**: lowest favourite win rate (~27%) → more races reach affordable longer-odds runners → higher per-bet growth → marginally better success

### Practical Implications
- The Betfair minimum back-stake (\$2) acts as a natural circuit breaker at very short odds
- As balance grows from \$5 → \$20 → \$100 → \$500, the selection automatically upgrades from favourite to 2nd/3rd/4th favourite
- The median session ends after **3–5 races** (bust at the first or second loss)
- This is a **high-risk, lottery-style** strategy: ~0.1–0.5% chance of a 200× return